# LIMINA -- 02. Preprocessing, Label, dan Normalisasi

Mengubah enam tabel mentah (`data/raw/`, hasil notebook 01) menjadi:

- `data/panel.csv` -- data latih, sampel berimbang 1 positif : 3 negatif
- `data/snapshot_<tanggal>.csv` -- enam potret evaluasi, seluruh emiten, proporsi kejadian apa adanya
- `data/jendela_latih.json` -- cutoff dan tanggal potret siklus ini, dibaca ulang notebook 03/04

Standardisasi (StandardScaler) TIDAK difinalkan di sini -- scaler produksi
dilatih di notebook 03, hanya pada data latih (AMBA-struktur-model-dan-
algoritma.md 3.4: scaler dilatih di atas data yang sudah tercampur potret
uji adalah kebocoran). Bagian akhir notebook ini hanya pratinjau visual.


In [1]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'limina/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek LIMINA."
    )


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import json
from datetime import datetime, timezone

import pandas as pd
from sklearn.preprocessing import StandardScaler

from limina import config, contracts, labels, raw_ingest, splits, supabase_io

df_qf = pd.read_csv(config.RAW_DIR / f"{config.TABEL_QUARTERLY_FINANCIALS}.csv")
df_dt = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_TRANSACTION}.csv")
df_dfu = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_FULL_UNIVERSE_CLOSE}.csv")
df_ff = pd.read_csv(config.RAW_DIR / f"{config.TABEL_FREE_FLOAT_SNAPSHOT}.csv")
df_co = pd.read_csv(config.RAW_DIR / f"{config.TABEL_COMPANY_OVERVIEW}.csv")
df_suspensi_raw = pd.read_csv(config.RAW_DIR / f"{config.TABEL_SUSPENSI}.csv")

# Validasi skema OTOMATIS untuk kelima tabel + normalisasi suspensi di
# bawah -- diperiksa di sini juga, tidak cuma mengandalkan notebook 01.
for _nama, _df in [
    (config.TABEL_QUARTERLY_FINANCIALS, df_qf),
    (config.TABEL_DAILY_TRANSACTION, df_dt),
    (config.TABEL_DAILY_FULL_UNIVERSE_CLOSE, df_dfu),
    (config.TABEL_FREE_FLOAT_SNAPSHOT, df_ff),
    (config.TABEL_COMPANY_OVERVIEW, df_co),
]:
    supabase_io.validasi_kolom_tabel(_df, _nama)

# Idempoten (supabase_io.py::normalisasi_tabel_suspensi) -- aman dipanggil
# lagi walau data/raw/*.csv datang dari luar notebook 01 (unggah manual, cache lama).
df_suspensi_raw = supabase_io.normalisasi_tabel_suspensi(df_suspensi_raw)

print(f"quarterly_financials : {len(df_qf):>8,} baris")
print(f"harga (2 tabel)       : {len(df_dt) + len(df_dfu):>8,} baris")
print(f"free_float_snapshot   : {len(df_ff):>8,} baris")
print(f"company_overview      : {len(df_co):>8,} baris")
print(f"stock_suspensions     : {len(df_suspensi_raw):>8,} baris (kolom: {list(df_suspensi_raw.columns)})")

quarterly_financials :       76 baris
harga (2 tabel)       :    4,131 baris
free_float_snapshot   :    2,883 baris
company_overview      :       19 baris
stock_suspensions     :      591 baris (kolom: ['symbol', 'event_date', 'reason', 'pdf_url', 'fetched_at', 'raw_response'])


## 1. Klasifikasi alasan suspensi

In [2]:
taksonomi = labels.muat_taksonomi()
df_suspensi_raw["event_category"] = df_suspensi_raw["reason"].apply(
    lambda a: labels.klasifikasi_alasan(a, taksonomi)
)

print(df_suspensi_raw["event_category"].value_counts(dropna=False))

tak_terklasifikasi = df_suspensi_raw[df_suspensi_raw["event_category"].isna()]
if len(tak_terklasifikasi):
    print(f"\n{len(tak_terklasifikasi)} baris tidak cocok kata kunci manapun, contoh:")
    print(tak_terklasifikasi[["symbol", "reason"]].head(10).to_string(index=False))
    print(
        "\nTinjau baris di atas secara manual. Kalau ternyata pola sah yang "
        "belum tercakup, tambahkan kata kuncinya ke "
        "data/labels/taksonomi_alasan_suspensi.json."
    )

df_suspensi_c = df_suspensi_raw[df_suspensi_raw["event_category"] == labels.KATEGORI_LABEL_POSITIF].copy()
print(f"\n{len(df_suspensi_c)} peristiwa kategori C (label positif) dari {len(df_suspensi_raw)} total")

event_category
B      467
C       64
NaN     56
A        4
Name: count, dtype: int64

56 baris tidak cocok kata kunci manapun, contoh:
 symbol                    reason
PURE.JK Suspend more than 6 month
MAGP.JK Suspend more than 6 month
WMPP.JK Suspend more than 6 month
DUCK.JK Suspend more than 6 month
GLOB.JK Suspend more than 6 month
SRIL.JK Suspend more than 6 month
TDPM.JK Suspend more than 6 month
UNIT.JK Suspend more than 6 month
POSA.JK Suspend more than 6 month
MTRA.JK Suspend more than 6 month

Tinjau baris di atas secara manual. Kalau ternyata pola sah yang belum tercakup, tambahkan kata kuncinya ke data/labels/taksonomi_alasan_suspensi.json.

64 peristiwa kategori C (label positif) dari 591 total


## 2. Peta sektor, papan pencatatan, dan cakupan emiten

In [3]:
peta_sektor = raw_ingest.bangun_peta_sektor(df_ff, df_co)
peta_board = raw_ingest.bangun_peta_board(df_co)
df_harga = raw_ingest.gabungkan_harga(df_dt, df_dfu)

# Cakupan DIBATASI ke emiten yang punya quarterly_financials DAN harga --
# BUKAN union dengan company_overview (yang mendaftar seluruh emiten
# tercatat, termasuk yang belum punya data mentah sama sekali; union
# hanya menambah baris data_complete=0, lihat symbols_dengan_data_lengkap).
symbols_universe = raw_ingest.symbols_dengan_data_lengkap(df_qf, df_harga)
symbols_tercatat_tanpa_data = sorted(
    (set(df_co["symbol"]) if len(df_co) else set()) - set(symbols_universe)
)
print(f"Cakupan emiten (quarterly_financials + harga tersedia): {len(symbols_universe)}")
print(f"Emiten tercatat di company_overview tapi belum punya data mentah: {len(symbols_tercatat_tanpa_data)}")
print(f"Emiten dengan board diketahui (dari company_overview): {len(peta_board)}")
print(f"Emiten dengan sector diketahui: {len(peta_sektor)}")

Cakupan emiten (quarterly_financials + harga tersedia): 19
Emiten tercatat di company_overview tapi belum punya data mentah: 0
Emiten dengan board diketahui (dari company_overview): 19
Emiten dengan sector diketahui: 19


## 3. Diagnosa cakupan data mentah

Cek dulu tumpang tindih `symbols_universe` dengan `quarterly_financials`/harga, dan rentang tanggalnya, sebelum membangun panel -- kalau kurang, `data_complete` akan 0 untuk sebagian besar baris di langkah berikutnya.

In [4]:
diagnosa = raw_ingest.diagnosa_cakupan_mentah(symbols_universe, df_qf, df_harga, df_suspensi_c)
for k, v in diagnosa.items():
    print(f"{k}: {v}")

if diagnosa["tumpang_tindih_quarterly_financials_persen"] < 90 or diagnosa["tumpang_tindih_harga_persen"] < 90:
    print(
        "\nPERINGATAN: tumpang tindih symbol < 90%. Cek format symbol "
        "(mis. akhiran '.JK') di contoh_symbol_universe vs "
        "contoh_symbol_quarterly_financials/contoh_symbol_harga di atas."
    )

if diagnosa.get("jumlah_symbol_kategori_c_siap_dilatih", 0) == 0 and diagnosa.get("jumlah_symbol_kategori_c", 0) > 0:
    print(
        "\nPERINGATAN: tidak ada symbol kategori C yang qf+harga-nya "
        "menjangkau tanggal peristiwanya -- panel akan 0 baris positif "
        "lengkap. Ini bukan bug, tapi cakupan data. Lihat tabel prioritas "
        "backfill di bagian 4 untuk target termurah."
    )

jumlah_symbols_universe: 19
tumpang_tindih_quarterly_financials_persen: 100.0
tumpang_tindih_harga_persen: 100.0
contoh_symbol_universe: ['AADI.JK', 'AMMN.JK', 'ASII.JK', 'BBCA.JK', 'BELI.JK']
contoh_symbol_quarterly_financials: ['AADI.JK', 'AMMN.JK', 'ASII.JK', 'BBCA.JK', 'BELI.JK']
contoh_symbol_harga: ['AADI.JK', 'AALI.JK', 'ABBA.JK', 'ABDA.JK', 'ABMM.JK']
report_date_min: 2024-12-31
report_date_max: 2026-06-30
harga_date_min: 2026-06-15
harga_date_max: 2026-09-16
jumlah_symbol_kategori_c: 46
jumlah_symbol_kategori_c_dengan_quarterly_financials: 1
symbol_kategori_c_dengan_quarterly_financials: ['MGLV.JK']
jumlah_symbol_kategori_c_siap_dilatih: 0
symbol_kategori_c_belum_punya_quarterly_financials_contoh: ['AKKU.JK', 'ALTO.JK', 'AMMS.JK', 'ASLI.JK', 'BCIC.JK', 'BEBS.JK', 'BIMA.JK', 'COAL.JK', 'DADA.JK', 'DART.JK', 'DPNS.JK', 'FASW.JK', 'FIMP.JK', 'FISH.JK', 'GGRP.JK']

PERINGATAN: tidak ada symbol kategori C yang qf+harga-nya menjangkau tanggal peristiwanya -- panel akan 0 baris posit

## 4. Jendela latih/evaluasi bergulir

Dihitung ULANG setiap kali cell ini jalan, relatif ke hari ini -- bukan
tanggal tetap. Ini yang membuat siklus latih besok otomatis bergeser maju
tanpa menyunting kode apa pun (lihat `limina/splits.py`).

In [5]:
cutoff_latih, tanggal_potret = splits.hitung_jendela_bergulir()
batas_matang = splits.batas_label_matang()

print(f"Dijalankan pada     : {datetime.now(timezone.utc).isoformat()}")
print(f"Batas label matang  : {batas_matang.date()}")
print(f"Cutoff latih        : {cutoff_latih.date()}")
print(f"Tanggal potret (6)  : {tanggal_potret}")

config.DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(config.PATH_JENDELA, "w", encoding="utf-8") as f:
    json.dump(
        {
            "dihitung_pada": datetime.now(timezone.utc).isoformat(),
            "cutoff_latih": cutoff_latih.strftime("%Y-%m-%d"),
            "tanggal_potret": tanggal_potret,
        },
        f,
        indent=2,
    )

# Prioritas backfill: symbol kategori C tanpa qf+harga lengkap, cocok_untuk_latih dulu, lalu termurah.
prioritas_backfill = raw_ingest.prioritas_backfill_kategori_c(
    df_suspensi_c, df_qf, df_harga, cutoff_latih=cutoff_latih
)
if len(prioritas_backfill):
    print("\nPrioritas backfill (cocok_untuk_latih dulu, lalu termurah):")
    display(prioritas_backfill.head(15))

Dijalankan pada     : 2026-09-20T23:47:27.416597+00:00
Batas label matang  : 2026-08-21
Cutoff latih        : 2026-03-24
Tanggal potret (6)  : ['2026-03-24', '2026-04-23', '2026-05-23', '2026-06-22', '2026-07-22', '2026-08-21']

Prioritas backfill (cocok_untuk_latih dulu, lalu termurah):


,symbol,event_date,sudah_punya_quarterly_financials,sudah_punya_harga,harga_awal_dibutuhkan,cocok_untuk_latih
0,ZBRA.JK,2026-01-22,False,True,2025-09-09,True
1,INRU.JK,2025-12-17,False,True,2025-08-04,True
2,MTPS.JK,2025-11-05,False,True,2025-06-23,True
3,DPNS.JK,2025-10-31,False,True,2025-06-18,True
4,FASW.JK,2025-09-30,False,True,2025-05-18,True
5,KIAS.JK,2025-09-30,False,True,2025-05-18,True
6,LMSH.JK,2025-09-30,False,True,2025-05-18,True
7,MTSM.JK,2025-09-30,False,True,2025-05-18,True
8,MFMI.JK,2025-09-30,False,True,2025-05-18,True
9,PLIN.JK,2025-09-30,False,True,2025-05-18,True


## 5. Bangun enam potret evaluasi (label sungguhan)

In [6]:
snapshot_dict = {}
for tanggal in tanggal_potret:
    snap = raw_ingest.bangun_snapshot_pasar(
        tanggal, symbols_universe, df_qf, df_harga, peta_sektor,
        peta_board=peta_board, df_suspensi_c=df_suspensi_c, taksonomi=taksonomi,
    )
    contracts.validate_panel(snap, ketat=True)  # melempar error kalau kontrak dilanggar
    snap.to_csv(config.path_snapshot(tanggal), index=False)
    snapshot_dict[tanggal] = snap
    print(
        f"potret {tanggal}: {len(snap):>5} emiten, "
        f"{int(snap['is_event_90d'].sum())} peristiwa dalam {config.JENDELA_LABEL_HARI} hari, "
        f"{int((snap['data_complete'] == 0).sum())} tidak lengkap"
    )

potret 2026-03-24:    19 emiten, 0 peristiwa dalam 30 hari, 19 tidak lengkap
potret 2026-04-23:    19 emiten, 0 peristiwa dalam 30 hari, 19 tidak lengkap


potret 2026-05-23:    19 emiten, 0 peristiwa dalam 30 hari, 19 tidak lengkap
potret 2026-06-22:    19 emiten, 0 peristiwa dalam 30 hari, 19 tidak lengkap


potret 2026-07-22:    19 emiten, 0 peristiwa dalam 30 hari, 0 tidak lengkap


potret 2026-08-21:    19 emiten, 0 peristiwa dalam 30 hari, 0 tidak lengkap


## 6. Bangun panel latih (sampel berimbang)

In [7]:
panel, ringkasan_panel = raw_ingest.bangun_panel_latih(
    df_suspensi_raw, symbols_universe, df_qf, df_harga, peta_sektor,
    taksonomi=taksonomi, peta_board=peta_board,
    kecualikan_tanggal=tanggal_potret, batas_akhir_as_of=batas_matang,
)
contracts.validate_panel(panel, ketat=True)
panel.to_csv(config.PATH_PANEL, index=False)

print(ringkasan_panel)
print(f"\nPanel disimpan: {config.PATH_PANEL} ({len(panel)} baris)")

if ringkasan_panel["total_kategori_c"] < 30:
    print(
        "\nPERINGATAN: <30 peristiwa kategori C -- selang kepercayaan metrik "
        "akan lebar (docs/rancangan/metodologi.md bagian 9, poin 1)."
    )

proporsi_lengkap = panel["data_complete"].mean() if len(panel) else 0.0
print(f"\nProporsi baris panel dengan data lengkap: {proporsi_lengkap:.1%}")
if proporsi_lengkap < 0.5:
    print(
        "PERINGATAN: <50% baris lengkap. Notebook 03 menyaring baris "
        "tidak lengkap sebelum melatih -- cek diagnosa bagian 3 dan "
        "tabel prioritas backfill bagian 4 kalau sisanya terlalu sedikit."
    )

{'total_suspensi_mentah': 591, 'total_kategori_c': 64, 'tak_terklasifikasi': 56, 'baris_panel': 256, 'positif_panel': 64}

Panel disimpan: /home/runner/work/LIMINA-model-train/LIMINA-model-train/data/panel.csv (256 baris)

Proporsi baris panel dengan data lengkap: 3.5%
PERINGATAN: <50% baris lengkap. Notebook 03 menyaring baris tidak lengkap sebelum melatih -- cek diagnosa bagian 3 dan tabel prioritas backfill bagian 4 kalau sisanya terlalu sedikit.


## 7. Pratinjau normalisasi (diagnostik, bukan scaler produksi)

Scaler yang benar-benar dipakai model dilatih ulang di notebook 03. Cell
berikut hanya untuk memeriksa sebaran fitur secara visual sebelum lanjut.

In [8]:
train_preview = splits.pisahkan_temporal(panel, cutoff_latih)
train_preview_lengkap = train_preview[train_preview["data_complete"] == 1]
X_preview = train_preview_lengkap[contracts.KOLOM_FITUR]

if len(X_preview) == 0:
    print(
        "Tidak ada baris data_complete == 1 -- pratinjau dilewati. "
        "Notebook 03 akan berhenti dengan penjelasan yang sama."
    )
else:
    print("Ringkasan fitur SEBELUM standardisasi (data latih, baris lengkap saja):")
    display(X_preview.describe().T[["mean", "std", "min", "max"]].round(3))

    pratinjau_scaler = StandardScaler()
    X_scaled_preview = pratinjau_scaler.fit_transform(X_preview.fillna(X_preview.median()))
    print("\nSetelah standardisasi, tiap kolom seharusnya mean sekitar 0, std sekitar 1:")
    display(
        pd.DataFrame(X_scaled_preview, columns=contracts.KOLOM_FITUR)
        .describe().T[["mean", "std"]].round(3)
    )

print(
    f"\nJumlah baris latih: {len(train_preview)} total, {len(X_preview)} data lengkap, "
    f"{int(train_preview['is_event_90d'].sum())} positif"
)
print("Lanjut ke notebook 03 (pelatihan_model).")

Ringkasan fitur SEBELUM standardisasi (data latih, baris lengkap saja):


,mean,std,min,max
lapor_jarak_hari,82.778,36.241,46.000,137.000
lapor_terlambat,0.556,0.527,0.000,1.000
tanpa_pendapatan,0.000,0.000,0.000,0.000
ekuitas_negatif,0.000,0.000,0.000,0.000
utang_terhadap_aset,0.636,0.188,0.359,0.857
ako_negatif_berturut,0.333,0.500,0.000,1.000
hari_tanpa_transaksi_90d,0.000,0.000,0.000,0.000
rasio_volume_30_90,1.000,0.000,1.000,1.000
hari_di_batas_bawah_90d,0.000,0.000,0.000,0.000
turun_dari_puncak_90d,-0.051,0.079,-0.188,0.000



Setelah standardisasi, tiap kolom seharusnya mean sekitar 0, std sekitar 1:


,mean,std
lapor_jarak_hari,0.0,1.061
lapor_terlambat,-0.0,1.061
tanpa_pendapatan,0.0,0.000
ekuitas_negatif,0.0,0.000
utang_terhadap_aset,0.0,1.061
ako_negatif_berturut,0.0,1.061
hari_tanpa_transaksi_90d,0.0,0.000
rasio_volume_30_90,0.0,0.000
hari_di_batas_bawah_90d,0.0,0.000
turun_dari_puncak_90d,-0.0,1.061



Jumlah baris latih: 249 total, 9 data lengkap, 57 positif
Lanjut ke notebook 03 (pelatihan_model).
